In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

batch_id = datetime.now().strftime("%Y%m%d%H%M%S")

print(f"Bronze to Silver started. Batch ID: {batch_id}")

# Create schemas
for schema_name in ["bronze", "silver", "gold", "quarantine"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

print("Schemas ready: bronze, silver, gold, quarantine")

StatementMeta(, 4b45a5cc-cb7b-454f-9b71-1910bbf90e40, 3, Finished, Available, Finished, False)

Bronze to Silver started. Batch ID: 20260714020515
Schemas ready: bronze, silver, gold, quarantine


In [2]:
def write_delta(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    print(f"{table_name}: {df.count()} rows")


def read_csv_with_metadata(source_name, path, load_type):
    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(path)
    )

    return (
        df
        .withColumn("_source_name", F.lit(source_name))
        .withColumn("_source_file_name", F.input_file_name())
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_batch_id", F.lit(batch_id))
        .withColumn("_load_type", F.lit(load_type))
    )


# Rebuild bronze.customers with schema-safe union
customers_initial = read_csv_with_metadata(
    "customers",
    "Files/retail_project/landing/customers/customers_initial.csv",
    "initial_raw"
)

customers_changes = read_csv_with_metadata(
    "customers",
    "Files/retail_project/landing/customers/customers_changes_2026_04.csv",
    "cdc_raw"
)

bronze_customers_fixed = customers_initial.unionByName(
    customers_changes,
    allowMissingColumns=True
)

write_delta(bronze_customers_fixed, "bronze.customers")


# Rebuild bronze.products with schema-safe union
products_initial = read_csv_with_metadata(
    "products",
    "Files/retail_project/landing/products/products_initial.csv",
    "initial_raw"
)

products_changes = read_csv_with_metadata(
    "products",
    "Files/retail_project/landing/products/products_changes_2026_04.csv",
    "cdc_raw"
)

bronze_products_fixed = products_initial.unionByName(
    products_changes,
    allowMissingColumns=True
)

write_delta(bronze_products_fixed, "bronze.products")
 

StatementMeta(, 4b45a5cc-cb7b-454f-9b71-1910bbf90e40, 4, Finished, Available, Finished, False)

bronze.customers: 129 rows
bronze.products: 38 rows


In [3]:
def clean_string_columns(df):
    """
    Trim all string columns and convert empty strings to null.
    """
    for col_name, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(col_name, F.trim(F.col(col_name)))
            df = df.withColumn(
                col_name,
                F.when(F.col(col_name) == "", None).otherwise(F.col(col_name))
            )
    return df


def ensure_column(df, col_name, data_type="string"):
    """
    Add a column if it does not exist.
    Useful when initial and CDC files have different columns.
    """
    if col_name not in df.columns:
        df = df.withColumn(col_name, F.lit(None).cast(data_type))
    return df


def deduplicate_latest(df, keys, order_cols):
    """
    Keep latest record based on keys and ordering columns.
    """
    window_spec = Window.partitionBy(*keys).orderBy(
        *[F.col(c).desc_nulls_last() for c in order_cols]
    )

    return (
        df
        .withColumn("_rn", F.row_number().over(window_spec))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )


def add_dq_error(df, rules):
    """
    Add data quality error messages.
    rules = list of tuples: (condition, message)
    """
    error_expressions = [
        F.when(condition, F.lit(message))
        for condition, message in rules
    ]

    return df.withColumn("_dq_error", F.concat_ws("; ", *error_expressions))


def split_valid_invalid(df, silver_table, quarantine_table):
    """
    Write valid rows to Silver and invalid rows to Quarantine.
    """
    valid_df = df.filter(F.length(F.col("_dq_error")) == 0)
    invalid_df = df.filter(F.length(F.col("_dq_error")) > 0)

    write_delta(valid_df, silver_table)
    write_delta(invalid_df, quarantine_table)

    print(f"Valid rows written to: {silver_table}")
    print(f"Invalid rows written to: {quarantine_table}")

StatementMeta(, 4b45a5cc-cb7b-454f-9b71-1910bbf90e40, 5, Finished, Available, Finished, False)

In [4]:
stores = spark.table("bronze.stores")
stores = clean_string_columns(stores)

stores_clean = (
    stores
    .withColumn("open_date", F.to_date("open_date"))
)

stores_clean = deduplicate_latest(
    stores_clean,
    keys=["store_id"],
    order_cols=["open_date", "_ingestion_timestamp"]
)

stores_clean = add_dq_error(
    stores_clean,
    [
        (F.col("store_id").isNull(), "store_id is missing"),
        (F.col("store_name").isNull(), "store_name is missing"),
        (F.col("country").isNull(), "country is missing"),
        (F.col("store_type").isNull(), "store_type is missing")
    ]
)

split_valid_invalid(
    stores_clean,
    "silver.stores",
    "quarantine.stores"
)

StatementMeta(, 4b45a5cc-cb7b-454f-9b71-1910bbf90e40, 6, Finished, Available, Finished, False)

silver.stores: 8 rows
quarantine.stores: 0 rows
Valid rows written to: silver.stores
Invalid rows written to: quarantine.stores


In [5]:
customers = spark.table("bronze.customers")
customers = clean_string_columns(customers)

customers = ensure_column(customers, "change_type")
customers = ensure_column(customers, "effective_date")

customers_clean = (
    customers
    .withColumn("email", F.lower(F.col("email")))
    .withColumn("created_at", F.to_date("created_at"))
    .withColumn("updated_at", F.to_date("updated_at"))
    .withColumn("effective_date", F.to_date("effective_date"))
    .withColumn(
        "change_type",
        F.when(F.col("change_type").isNull(), F.lit("INITIAL"))
         .otherwise(F.upper(F.col("change_type")))
    )
    .withColumn(
        "effective_date",
        F.coalesce(F.col("effective_date"), F.col("updated_at"), F.col("created_at"))
    )
)

customers_clean = deduplicate_latest(
    customers_clean,
    keys=["customer_id", "effective_date", "change_type"],
    order_cols=["updated_at", "_ingestion_timestamp"]
)

customers_clean = add_dq_error(
    customers_clean,
    [
        (F.col("customer_id").isNull(), "customer_id is missing"),
        (F.col("first_name").isNull(), "first_name is missing"),
        (F.col("last_name").isNull(), "last_name is missing"),
        (
            F.col("email").isNull() | (~F.col("email").rlike("^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$")),
            "email is invalid"
        ),
        (F.col("city").isNull(), "city is missing"),
        (F.col("country").isNull(), "country is missing"),
        (F.col("effective_date").isNull(), "effective_date is missing")
    ]
)

split_valid_invalid(
    customers_clean,
    "silver.customers",
    "quarantine.customers"
)

StatementMeta(, 4b45a5cc-cb7b-454f-9b71-1910bbf90e40, 7, Finished, Available, Finished, False)

silver.customers: 127 rows
quarantine.customers: 1 rows
Valid rows written to: silver.customers
Invalid rows written to: quarantine.customers


In [6]:
products = spark.table("bronze.products")
products = clean_string_columns(products)

products = ensure_column(products, "change_type")
products = ensure_column(products, "effective_date")

products_clean = (
    products
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn("cost_price", F.col("cost_price").cast("double"))
    .withColumn("is_active", F.col("is_active").cast("boolean"))
    .withColumn("created_at", F.to_date("created_at"))
    .withColumn("updated_at", F.to_date("updated_at"))
    .withColumn("effective_date", F.to_date("effective_date"))
    .withColumn(
        "change_type",
        F.when(F.col("change_type").isNull(), F.lit("INITIAL"))
         .otherwise(F.upper(F.col("change_type")))
    )
    .withColumn(
        "effective_date",
        F.coalesce(F.col("effective_date"), F.col("updated_at"), F.col("created_at"))
    )
)

products_clean = deduplicate_latest(
    products_clean,
    keys=["product_id", "effective_date", "change_type"],
    order_cols=["updated_at", "_ingestion_timestamp"]
)

products_clean = add_dq_error(
    products_clean,
    [
        (F.col("product_id").isNull(), "product_id is missing"),
        (F.col("product_name").isNull(), "product_name is missing"),
        (F.col("category").isNull(), "category is missing"),
        (F.col("unit_price").isNull(), "unit_price is missing"),
        (F.col("unit_price") <= 0, "unit_price must be greater than 0"),
        (F.col("cost_price").isNull(), "cost_price is missing"),
        (F.col("cost_price") < 0, "cost_price cannot be negative"),
        (F.col("effective_date").isNull(), "effective_date is missing")
    ]
)

split_valid_invalid(
    products_clean,
    "silver.products",
    "quarantine.products"
)

StatementMeta(, 4b45a5cc-cb7b-454f-9b71-1910bbf90e40, 8, Finished, Available, Finished, False)

silver.products: 36 rows
quarantine.products: 1 rows
Valid rows written to: silver.products
Invalid rows written to: quarantine.products


In [7]:
orders = spark.table("bronze.orders")
orders = clean_string_columns(orders)

orders_clean = (
    orders
    .withColumn("order_date", F.to_timestamp("order_date"))
    .withColumn("shipping_date", F.to_date("shipping_date"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn("discount_amount", F.coalesce(F.col("discount_amount").cast("double"), F.lit(0.0)))
    .withColumn("created_at", F.to_timestamp("created_at"))
    .withColumn("updated_at", F.to_timestamp("updated_at"))
    .withColumn("channel", F.initcap(F.col("channel")))
)

# Create a safe deduplication key so missing order_id records are not all collapsed together
orders_clean = orders_clean.withColumn(
    "_dedup_key",
    F.when(
        F.col("order_id").isNull(),
        F.concat(F.lit("__missing_order_id_"), F.monotonically_increasing_id())
    ).otherwise(F.col("order_id"))
)

orders_clean = deduplicate_latest(
    orders_clean,
    keys=["_dedup_key"],
    order_cols=["updated_at", "_ingestion_timestamp"]
)

valid_customers = (
    spark.table("silver.customers")
    .select("customer_id")
    .distinct()
    .withColumn("_valid_customer", F.lit(1))
)

valid_products = (
    spark.table("silver.products")
    .select("product_id")
    .distinct()
    .withColumn("_valid_product", F.lit(1))
)

valid_stores = (
    spark.table("silver.stores")
    .select("store_id")
    .distinct()
    .withColumn("_valid_store", F.lit(1))
)

orders_clean = (
    orders_clean
    .join(valid_customers, on="customer_id", how="left")
    .join(valid_products, on="product_id", how="left")
    .join(valid_stores, on="store_id", how="left")
)

orders_clean = add_dq_error(
    orders_clean,
    [
        (F.col("order_id").isNull(), "order_id is missing"),
        (F.col("order_date").isNull(), "order_date is invalid"),
        (F.col("order_date") > F.current_timestamp(), "order_date is in the future"),
        (F.col("customer_id").isNull(), "customer_id is missing"),
        (F.col("product_id").isNull(), "product_id is missing"),
        (F.col("store_id").isNull(), "store_id is missing"),
        (F.col("quantity").isNull(), "quantity is missing"),
        (F.col("quantity") <= 0, "quantity must be greater than 0"),
        (F.col("unit_price").isNull(), "unit_price is missing"),
        (F.col("unit_price") <= 0, "unit_price must be greater than 0"),
        (F.col("customer_id").isNotNull() & F.col("_valid_customer").isNull(), "customer_id does not exist"),
        (F.col("product_id").isNotNull() & F.col("_valid_product").isNull(), "product_id does not exist"),
        (F.col("store_id").isNotNull() & F.col("_valid_store").isNull(), "store_id does not exist")
    ]
)

orders_clean = orders_clean.drop(
    "_valid_customer",
    "_valid_product",
    "_valid_store",
    "_dedup_key"
)

split_valid_invalid(
    orders_clean,
    "silver.orders",
    "quarantine.orders"
)

StatementMeta(, 4b45a5cc-cb7b-454f-9b71-1910bbf90e40, 9, Finished, Available, Finished, False)

silver.orders: 1206 rows
quarantine.orders: 5 rows
Valid rows written to: silver.orders
Invalid rows written to: quarantine.orders


In [8]:
inventory = spark.table("bronze.inventory")
inventory = clean_string_columns(inventory)

inventory_clean = (
    inventory
    .withColumn("snapshot_date", F.to_date("snapshot_date"))
    .withColumn("on_hand_qty", F.col("on_hand_qty").cast("int"))
    .withColumn("reserved_qty", F.col("reserved_qty").cast("int"))
    .withColumn("reorder_level", F.col("reorder_level").cast("int"))
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

inventory_clean = deduplicate_latest(
    inventory_clean,
    keys=["snapshot_date", "store_id", "product_id"],
    order_cols=["updated_at", "_ingestion_timestamp"]
)

valid_products = (
    spark.table("silver.products")
    .select("product_id")
    .distinct()
    .withColumn("_valid_product", F.lit(1))
)

valid_stores = (
    spark.table("silver.stores")
    .select("store_id")
    .distinct()
    .withColumn("_valid_store", F.lit(1))
)

inventory_clean = (
    inventory_clean
    .join(valid_products, on="product_id", how="left")
    .join(valid_stores, on="store_id", how="left")
)

inventory_clean = add_dq_error(
    inventory_clean,
    [
        (F.col("snapshot_date").isNull(), "snapshot_date is missing"),
        (F.col("store_id").isNull(), "store_id is missing"),
        (F.col("product_id").isNull(), "product_id is missing"),
        (F.col("on_hand_qty").isNull(), "on_hand_qty is missing"),
        (F.col("on_hand_qty") < 0, "on_hand_qty cannot be negative"),
        (F.col("reserved_qty").isNull(), "reserved_qty is missing"),
        (F.col("reserved_qty") < 0, "reserved_qty cannot be negative"),
        (F.col("reorder_level").isNull(), "reorder_level is missing"),
        (F.col("store_id").isNotNull() & F.col("_valid_store").isNull(), "store_id does not exist"),
        (F.col("product_id").isNotNull() & F.col("_valid_product").isNull(), "product_id does not exist")
    ]
)

inventory_clean = inventory_clean.drop("_valid_product", "_valid_store")

split_valid_invalid(
    inventory_clean,
    "silver.inventory",
    "quarantine.inventory"
)

StatementMeta(, 4b45a5cc-cb7b-454f-9b71-1910bbf90e40, 10, Finished, Available, Finished, False)

silver.inventory: 420 rows
quarantine.inventory: 2 rows
Valid rows written to: silver.inventory
Invalid rows written to: quarantine.inventory


In [9]:
returns = spark.table("bronze.returns")
returns = clean_string_columns(returns)

returns_clean = (
    returns
    .withColumn("return_date", F.to_date("return_date"))
    .withColumn("return_quantity", F.col("return_quantity").cast("int"))
    .withColumn("refund_amount", F.col("refund_amount").cast("double"))
    .withColumn("created_at", F.to_timestamp("created_at"))
)

returns_clean = deduplicate_latest(
    returns_clean,
    keys=["return_id"],
    order_cols=["created_at", "_ingestion_timestamp"]
)

valid_orders = (
    spark.table("silver.orders")
    .select("order_id")
    .distinct()
    .withColumn("_valid_order", F.lit(1))
)

valid_customers = (
    spark.table("silver.customers")
    .select("customer_id")
    .distinct()
    .withColumn("_valid_customer", F.lit(1))
)

valid_products = (
    spark.table("silver.products")
    .select("product_id")
    .distinct()
    .withColumn("_valid_product", F.lit(1))
)

returns_clean = (
    returns_clean
    .join(valid_orders, on="order_id", how="left")
    .join(valid_customers, on="customer_id", how="left")
    .join(valid_products, on="product_id", how="left")
)

returns_clean = add_dq_error(
    returns_clean,
    [
        (F.col("return_id").isNull(), "return_id is missing"),
        (F.col("order_id").isNull(), "order_id is missing"),
        (F.col("product_id").isNull(), "product_id is missing"),
        (F.col("customer_id").isNull(), "customer_id is missing"),
        (F.col("return_date").isNull(), "return_date is invalid"),
        (F.col("return_quantity").isNull(), "return_quantity is missing"),
        (F.col("return_quantity") <= 0, "return_quantity must be greater than 0"),
        (F.col("refund_amount").isNull(), "refund_amount is missing"),
        (F.col("refund_amount") < 0, "refund_amount cannot be negative"),
        (F.col("order_id").isNotNull() & F.col("_valid_order").isNull(), "order_id does not exist"),
        (F.col("customer_id").isNotNull() & F.col("_valid_customer").isNull(), "customer_id does not exist"),
        (F.col("product_id").isNotNull() & F.col("_valid_product").isNull(), "product_id does not exist")
    ]
)

returns_clean = returns_clean.drop(
    "_valid_order",
    "_valid_customer",
    "_valid_product"
)

split_valid_invalid(
    returns_clean,
    "silver.returns",
    "quarantine.returns"
)

StatementMeta(, 4b45a5cc-cb7b-454f-9b71-1910bbf90e40, 11, Finished, Available, Finished, False)

silver.returns: 80 rows
quarantine.returns: 1 rows
Valid rows written to: silver.returns
Invalid rows written to: quarantine.returns


In [10]:
clickstream = spark.table("bronze.clickstream")
clickstream = clean_string_columns(clickstream)

clickstream_clean = (
    clickstream
    .withColumn("event_timestamp", F.to_timestamp("event_timestamp"))
)

clickstream_clean = clickstream_clean.withColumn(
    "_dedup_key",
    F.when(
        F.col("event_id").isNull(),
        F.concat(F.lit("__missing_event_id_"), F.monotonically_increasing_id())
    ).otherwise(F.col("event_id"))
)

clickstream_clean = deduplicate_latest(
    clickstream_clean,
    keys=["_dedup_key"],
    order_cols=["event_timestamp", "_ingestion_timestamp"]
)

valid_customers = (
    spark.table("silver.customers")
    .select("customer_id")
    .distinct()
    .withColumn("_valid_customer", F.lit(1))
)

valid_products = (
    spark.table("silver.products")
    .select("product_id")
    .distinct()
    .withColumn("_valid_product", F.lit(1))
)

clickstream_clean = (
    clickstream_clean
    .join(valid_customers, on="customer_id", how="left")
    .join(valid_products, on="product_id", how="left")
)

clickstream_clean = add_dq_error(
    clickstream_clean,
    [
        (F.col("event_id").isNull(), "event_id is missing"),
        (F.col("event_timestamp").isNull(), "event_timestamp is invalid"),
        (F.col("event_timestamp") > F.current_timestamp(), "event_timestamp is in the future"),
        (F.col("event_type").isNull(), "event_type is missing"),
        (
            F.col("customer_id").isNotNull() & F.col("_valid_customer").isNull(),
            "customer_id does not exist"
        ),
        (
            F.col("product_id").isNotNull() & F.col("_valid_product").isNull(),
            "product_id does not exist"
        )
    ]
)

clickstream_clean = clickstream_clean.drop(
    "_valid_customer",
    "_valid_product",
    "_dedup_key"
)

split_valid_invalid(
    clickstream_clean,
    "silver.clickstream",
    "quarantine.clickstream"
)

StatementMeta(, 4b45a5cc-cb7b-454f-9b71-1910bbf90e40, 12, Finished, Available, Finished, False)

silver.clickstream: 725 rows
quarantine.clickstream: 25 rows
Valid rows written to: silver.clickstream
Invalid rows written to: quarantine.clickstream


In [11]:
tables_to_check = [
    "silver.stores",
    "silver.customers",
    "silver.products",
    "silver.orders",
    "silver.inventory",
    "silver.returns",
    "silver.clickstream",
    "quarantine.stores",
    "quarantine.customers",
    "quarantine.products",
    "quarantine.orders",
    "quarantine.inventory",
    "quarantine.returns",
    "quarantine.clickstream"
]

summary = []

for table_name in tables_to_check:
    row_count = spark.table(table_name).count()
    summary.append((table_name, row_count))

summary_df = spark.createDataFrame(summary, ["table_name", "row_count"])
display(summary_df)

StatementMeta(, 4b45a5cc-cb7b-454f-9b71-1910bbf90e40, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b839ccd8-fb47-4dc6-809c-b83cf4ea588e)

In [12]:
display(spark.table("silver.orders").limit(10))

StatementMeta(, 4b45a5cc-cb7b-454f-9b71-1910bbf90e40, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 45cc0b21-8a67-45ef-8e40-92fdf856cc85)